# Microsam Finetuning on C. albicans data

## View data and model paths

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

### Remove stale checkpoints if necessary

In [ ]:
# %rm -rf /kaggle/working/model_checkpoints/

## Install latest microsam

In [ ]:
# Install micro_sam (pulls torch_em + all deps), then overlay the fork that contains the
# multi-GPU AIS training script. --no-deps keeps the working torch / torch_em from the first install.
# Requires: Settings -> Internet = ON, Accelerator = GPU T4 x2.
!pip install -q micro_sam
%rm -rf /kaggle/working/micro-sam
!git clone -q -b feature/candida-multigpu-ais https://github.com/chuyaowang/micro-sam.git /kaggle/working/micro-sam
!pip install -q -e /kaggle/working/micro-sam --no-deps

SCRIPT_DIR = "/kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy"

## Data loading

In [ ]:
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Union
import cv2
import numpy as np
import tifffile
from scipy.ndimage import shift
import re
import os

# from image_processing_tools.util.load_files import find_dapi_channel_file

logger = logging.getLogger(__name__)

class _SingleChannel:
    """
    Private helper class to manage a single image file.
    Handles lazy loading and preprocessing for one channel.
    """

    def __init__(self, image_path: Path, config: Dict[str, Any], is_label: bool = False):
        self.path = image_path
        self.config = config
        self.is_label = is_label
        self.proc_config = self.config.get("preprocessing", {})
        self._image_16bit: np.ndarray | None = None
        self._image_8bit: np.ndarray | None = None
        self._resized_8bit: np.ndarray | None = None
        self._resized_16bit: np.ndarray | None = None
        self.scale_factor = 1.0

    @property
    def image_16bit(self) -> np.ndarray:
        if self._image_16bit is None:
            logger.debug(f"Lazily loading image from {self.path}")
            img = tifffile.imread(str(self.path))
            if img is None:
                raise IOError(f"Failed to load image from {self.path}")
            self._image_16bit = img.astype(np.uint16)

            dic_shift = self.proc_config.get("correct_DIC_shift")
            if sum(dic_shift) != 0 and "DIC" in self.path.name.upper():
                if isinstance(dic_shift, bool):
                    raise TypeError(f"Invalid `correct_DIC_shift` value: {dic_shift}. Expected a list of dimensions.")
                if self._image_16bit.ndim == 3:
                    # Use last 2 elements as [dy, dx] — works for both [dy, dx] and [dz, dy, dx] inputs
                    spatial_shift = list(dic_shift)[-2:]
                    logger.info(f"Correcting DIC shift for {self.path.name} per slice with shift {spatial_shift}")
                    for z in range(self._image_16bit.shape[0]):
                        self._image_16bit[z] = shift(self._image_16bit[z], shift=spatial_shift, mode='nearest')
                else:
                    logger.info(f"Correcting DIC shift for {self.path.name} with shift {list(dic_shift)}")
                    self._image_16bit = shift(self._image_16bit, shift=list(dic_shift), mode='nearest')
        return self._image_16bit

    def _get_high_contrast_16bit(self) -> np.ndarray:
        outlier_percentile = self.proc_config.get("outlier_percentile", 0.35)
        if outlier_percentile <= 0:
            return self.image_16bit

        img = self.image_16bit.copy().astype(np.float32)
        min_val, max_val = np.percentile(img, (outlier_percentile, 100 - outlier_percentile))

        if max_val > min_val:
            img = np.clip(img, min_val, max_val)
            img = ((img - min_val) / (max_val - min_val) * 65535).astype(np.uint16)
        else:
            img = np.zeros(img.shape, dtype=np.uint16)
        return img

    @property
    def image_8bit(self) -> np.ndarray:
        if self._image_8bit is None:
            img_16bit_hc = self._get_high_contrast_16bit()
            self._image_8bit = (img_16bit_hc / 257).astype(np.uint8)
        return self._image_8bit

    @property
    def resized_8bit(self) -> np.ndarray:
        if self._resized_8bit is None:
            image = self.image_8bit
            if self.proc_config.get("resize_image", True):
                max_dim = self.proc_config.get("max_dim", 1024)
                is_3d = image.ndim == 3 and image.shape[0] > 1
                height, width = (image.shape[1], image.shape[2]) if is_3d else image.shape[:2]
                longest_edge = max(height, width)
                if longest_edge != max_dim:
                    self.scale_factor = max_dim / longest_edge
                    new_h = int(height * self.scale_factor)
                    new_w = int(width * self.scale_factor)
                    interpolation = cv2.INTER_AREA if self.scale_factor < 1.0 else cv2.INTER_LINEAR
                    logger.info(f"Resizing 8-bit image from ({height},{width}) to ({new_h},{new_w}) (scale={self.scale_factor:.4f})")
                    if is_3d:
                        self._resized_8bit = np.stack(
                            [cv2.resize(image[z], (new_w, new_h), interpolation=interpolation) for z in range(image.shape[0])],
                            axis=0,
                        )
                    else:
                        self._resized_8bit = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
                    return self._resized_8bit
            self._resized_8bit = image
        return self._resized_8bit

    @property
    def resized_16bit(self) -> np.ndarray:
        if self._resized_16bit is None:
            image = self._get_high_contrast_16bit()
            if self.proc_config.get("resize_image", True):
                max_dim = self.proc_config.get("max_dim", 1024)
                is_3d = image.ndim == 3 and image.shape[0] > 1
                height, width = (image.shape[1], image.shape[2]) if is_3d else image.shape[:2]
                longest_edge = max(height, width)
                if longest_edge != max_dim:
                    self.scale_factor = max_dim / longest_edge
                    new_h = int(height * self.scale_factor)
                    new_w = int(width * self.scale_factor)
                    interpolation = cv2.INTER_AREA if self.scale_factor < 1.0 else cv2.INTER_LINEAR
                    logger.info(f"Resizing 16-bit image from ({height},{width}) to ({new_h},{new_w}) (scale={self.scale_factor:.4f})")
                    if is_3d:
                        self._resized_16bit = np.stack(
                            [cv2.resize(image[z], (new_w, new_h), interpolation=interpolation) for z in range(image.shape[0])],
                            axis=0,
                        )
                    else:
                        self._resized_16bit = cv2.resize(image, (new_w, new_h), interpolation=interpolation)
                    return self._resized_16bit
            self._resized_16bit = image
        return self._resized_16bit

    def get_image_for_processing(self) -> np.ndarray:
        if self.is_label:
            image = self.image_16bit
            if self.proc_config.get("resize_image", True):
                max_dim = self.proc_config.get("max_dim", 1024)
                is_3d = image.ndim == 3 and image.shape[0] > 1
                height, width = (image.shape[1], image.shape[2]) if is_3d else image.shape[:2]
                if max(height, width) != max_dim:
                    scale = max_dim / max(height, width)
                    new_h, new_w = int(height * scale), int(width * scale)
                    if is_3d:
                        image = np.stack(
                            [cv2.resize(image[z], (new_w, new_h), interpolation=cv2.INTER_NEAREST) for z in range(image.shape[0])],
                            axis=0,
                        )
                    else:
                        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
            return image

        quantization = self.proc_config.get("quantization", "8bit")
        resize = self.proc_config.get("resize_image", True)
        if quantization == "8bit":
            return self.resized_8bit if resize else self.image_8bit
        elif quantization == "16bit":
            return self.resized_16bit if resize else self._get_high_contrast_16bit()
        else:
            logger.warning(f"Unknown quantization '{quantization}'. Defaulting to 8-bit.")
            return self.resized_8bit if resize else self.image_8bit

    def set_processed_image(self, image: np.ndarray):
        """Directly sets image attributes, bypassing lazy loading."""
        is_resized = self.proc_config.get("resize_image", True)
        if image.dtype == np.uint16:
            if is_resized:
                # The summed image is a resized image.
                self._resized_16bit = image
                self.scale_factor = self.proc_config.get("max_dim", 1024) / max(image.shape)
            else:
                # The summed image is a full-size image.
                self._image_16bit = image
        elif image.dtype == np.uint8:
            if is_resized:
                # The summed image is a resized image.
                self._resized_8bit = image
                self.scale_factor = self.proc_config.get("max_dim", 1024) / max(image.shape)
            else:
                # The summed image is a full-size image.
                self._image_8bit = image


class ImageContainer:
    """
    A unified class to represent and process single or multi-channel images.

    This class handles lazy loading, preprocessing, composition, and prompt generation.
    It can be initialized with a single file path for a mono-channel image or a list
    of paths for a multi-channel composite.
    """

    def __init__(self, structure: Union[Path, List[Any]], config: Dict[str, Any], is_label: bool = False):
        self.config = config
        self.is_label = is_label
        self.proc_config = self.config.get("preprocessing", {})
        self._source_paths: List[Path] = []
        
        source_path_set = set()

        if isinstance(structure, Path):
            # Base case: initialize with a single path
            self.channels: List[_SingleChannel] = [_SingleChannel(structure, config, is_label=is_label)]
            source_path_set.add(structure)
        else:
            # Handle complex structures like [[c1, c2], c3]
            processed_channels = []
            for item in structure:
                if isinstance(item, list):
                    # This is a group to be summed.
                    channels_to_sum = []
                    for sub_item in item:
                        if isinstance(sub_item, ImageContainer):
                            source_path_set.update(sub_item._source_paths)
                            channels_to_sum.extend(sub_item.channels)
                        elif isinstance(sub_item, Path):
                            source_path_set.add(sub_item)
                            channels_to_sum.append(_SingleChannel(sub_item, config, is_label=is_label))
                        else:
                            raise TypeError(f"Unsupported type in summation list: {type(sub_item)}")

                    if channels_to_sum:
                        summed_channel = self._sum_channels(channels_to_sum)
                        processed_channels.append(summed_channel)

                elif isinstance(item, ImageContainer):
                    source_path_set.update(item._source_paths)
                    processed_channels.extend(item.channels)
                elif isinstance(item, Path):
                    source_path_set.add(item)
                    processed_channels.append(_SingleChannel(item, config, is_label=is_label))
                else:
                    raise TypeError(f"Unsupported type in ImageContainer structure: {type(item)}")
            self.channels = processed_channels
        
        self._source_paths = list(source_path_set)

    @property
    def name(self) -> str:
        """
        Returns a descriptive name for the container.
        - If it contains one channel, it returns the channel's filename.
        - If it contains multiple channels, it creates a composite name.
        """
        # --- Helper function to get the short name (e.g., 'DAPI') from a path ---
        def get_channel_short_name(p: Path) -> str:
            try:
                # e.g., from "..._CY5,FITC,DAPI.tif" -> "CY5,FITC,DAPI"
                channel_list_str = p.name.rsplit('_', 1)[-1].rsplit('.', 1)[0]
                channels = [ch.strip() for ch in channel_list_str.split(',')]
                
                # Find which C-number this path corresponds to, e.g., "C3"
                match = re.search(r'C(\d+)', p.name)
                if match:
                    channel_num = int(match.group(1))
                    if 1 <= channel_num <= len(channels):
                        return channels[channel_num - 1] # 1-based to 0-based index
            except (IndexError, ValueError):
                pass # Fallback if parsing fails
            return p.stem # Fallback to the stem if a short name can't be parsed

        # --- Find all unique source paths that make up this container ---
        # This is now reliably populated by the constructor.
        if not self._source_paths:
            return "+".join(ch.path.stem for ch in self.channels)

        # --- Generate the name ---
        if len(self._source_paths) == 1:
            return self._source_paths[0].stem

        # Find the common base name by removing channel-specific parts (_C1_, _C2_, etc.)
        base_name = re.sub(r'_C\d_', '_', self._source_paths[0].stem)

        # --- Generate the structure part of the name (e.g., "CY5+FITC,DAPI") ---
        structure_parts = []
        for ch in self.channels:
            # The stem can be a single stem or a "+"-joined string of stems for summed channels.
            stems = ch.path.stem.split('+')
            part_names = []
            for stem in stems:
                # Find the original Path object from the complete list of source paths.
                found = False
                for p in self._source_paths:
                    if p.stem == stem:
                        part_names.append(get_channel_short_name(p))
                        found = True
                        break
                if not found:
                    # This fallback should ideally not be reached if __init__ is correct.
                    part_names.append(stem)
            structure_parts.append("+".join(part_names))
        
        structure_string = ','.join(structure_parts)
        return f"{base_name}_{structure_string}"

    def _sum_channels(self, channel_group: List[_SingleChannel]) -> _SingleChannel:
        """Sums a group of channels into a single new _SingleChannel object."""
        images_to_sum = [ch.get_image_for_processing() for ch in channel_group]
        summed_image_float = np.sum(np.stack(images_to_sum, axis=0).astype(np.float64), axis=0)

        # Create an informative name by joining the stems of the channels being summed.
        summed_channel_name = "+".join([ch.path.stem for ch in channel_group])
        # Use the config of the first channel for the config of the summed channel.
        summed_channel = _SingleChannel(Path(summed_channel_name), channel_group[0].config)

        dtype = images_to_sum[0].dtype
        d_info = np.iinfo(dtype)
        min_val, max_val = summed_image_float.min(), summed_image_float.max()

        if max_val > min_val:
            summed_image = ((summed_image_float - min_val) / (max_val - min_val) * d_info.max).astype(dtype)
        else:
            summed_image = np.zeros_like(summed_image_float, dtype=dtype)

        summed_channel.set_processed_image(summed_image)
        return summed_channel

    def __add__(self, other: "ImageContainer") -> "ImageContainer":
        """
        Combines two ImageContainer objects by creating a new container
        that holds the structure of both. For example, if `self` represents
        `[c1, c3]` and `other` represents `[c2]`, the result is equivalent to
        `ImageContainer([[c1, c3], c2], config)`.
        """
        return ImageContainer([self, other], self.config)

    def __getitem__(self, index: int) -> _SingleChannel:
        """Allows accessing individual channels by index."""
        return self.channels[index]

    def _is_3d(self) -> bool:
        """Returns True if the first channel is a multi-slice z-stack."""
        if not self.channels:
            return False
        try:
            return self.channels[0].image_16bit.ndim == 3
        except Exception:
            return False

    def _merge_3d(self) -> np.ndarray:
        """
        Merges 3D channel volumes into (Z, H, W, 3) for SAM.
        - 1 channel  → replicate × 3
        - 2 channels → average, then replicate × 3
        - 3 channels → use as-is
        - 4+ channels → ch1, ch2 unchanged; ch3..chN averaged into ch3
        """
        volumes = [ch.get_image_for_processing() for ch in self.channels]
        n = len(volumes)

        if n == 1:
            ch = volumes[0]
            result = np.stack([ch, ch, ch], axis=-1)
            logger.info("3D merge: 1 channel replicated to (Z, H, W, 3).")
        elif n == 2:
            merge_mode = self.config.get("preprocessing", {}).get("two_channel_merge_mode", "average_replicate")
            if merge_mode == "passthrough":
                result = np.stack(volumes, axis=-1)
                logger.info("3D merge: 2 channels passed through as (Z, H, W, 2); micro_sam will pad zero blue channel.")
            else:
                avg = ((volumes[0].astype(np.float32) + volumes[1].astype(np.float32)) / 2).astype(np.uint16)
                result = np.stack([avg, avg, avg], axis=-1)
                logger.info("3D merge: 2 channels averaged and replicated to (Z, H, W, 3).")
        elif n == 3:
            result = np.stack(volumes, axis=-1)
            logger.info("3D merge: 3 channels stacked to (Z, H, W, 3).")
        else:
            ch3 = np.mean(np.stack([v.astype(np.float32) for v in volumes[2:]], axis=0), axis=0).astype(np.uint16)
            result = np.stack([volumes[0], volumes[1], ch3], axis=-1)
            logger.info(f"3D merge: {n} channels → (Z, H, W, 3), ch3..ch{n} averaged into ch3.")

        return result

    def merge(self) -> np.ndarray:
        """
        Merges the channel(s) into a single NumPy array.

        For 3D z-stacks (auto-detected from file shape): returns (Z, H, W, 3).
        For 2D images:
          - 1 channel  → (H, W)
          - >1 channels → (H, W, C) via cv2.merge
        """
        if not self.channels:
            raise ValueError("Cannot merge an empty ImageContainer.")

        if self._is_3d():
            return self._merge_3d()

        images = [ch.get_image_for_processing() for ch in self.channels]

        if len(images) == 1:
            logger.info("Returning single channel image.")
            return images[0]

        if len(set(img.shape for img in images)) > 1:
            logger.warning("Channel images have different shapes. Resizing to the first channel's shape.")
            target_shape = images[0].shape
            images = [cv2.resize(img, (target_shape[1], target_shape[0])) if img.shape != target_shape else img for img in images]

        merged_image = cv2.merge(images)
        logger.info(f"Merged {len(self.channels)} channels into a {merged_image.shape} image.")
        return merged_image

    def _normalize_for_display(self, image: np.ndarray) -> np.ndarray:
        """Normalizes an image to a displayable format (float 0-1)."""
        if image.dtype == np.uint16:
            return image.astype(np.float32) / 65535.0
        elif image.dtype == np.uint8:
            return image
        
        min_val, max_val = image.min(), image.max()
        if max_val > 1.0:
            return (image - min_val) / (max_val - min_val)
        return image

    def display(self, title: str = "Image", cmap: str = "gray", ax=None):
        """Displays the merged image, handling 1, 2, or 3 channels correctly."""
        import matplotlib.pyplot as plt

        merged_image = self.merge()
        
        if merged_image.ndim == 3:
            num_ch = merged_image.shape[2]
            if num_ch == 2:
                logger.info("Displaying 2-channel image by mapping to Red and Green.")
                blue_channel = np.zeros_like(merged_image[:, :, 0])
                display_image = cv2.merge([merged_image[:, :, 0], merged_image[:, :, 1], blue_channel])
            elif num_ch == 3:
                display_image = merged_image
            else:
                logger.warning(f"Displaying first channel of {num_ch}-channel image.")
                display_image = merged_image[:, :, 0]
        else: # Grayscale
            display_image = merged_image

        # display_image = self._normalize_for_display(display_image)

        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 8))
            show = True
        else:
            show = False

        cmap = cmap if display_image.ndim == 2 else None
        ax.imshow(display_image, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")

        if show:
            plt.show()


In [ ]:
from typing import Union, Dict, Any
from pathlib import Path
import json

def load_config(config_path: Union[str, Path]) -> Dict[str, Any]:
    """
    Loads a JSON configuration file.

    Args:
        config_path (Union[str, Path]): The path to the JSON config file.

    Returns:
        Dict[str, Any]: A dictionary containing the configuration parameters.
    """
    config_path = Path(config_path)
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found at: {config_path}")

    with open(config_path, 'r') as f:
        config = json.load(f)
    return config

data_config = "/kaggle/input/datasets/gudagudao/microsam-finetune-1/microsam_finetune_data_kaggle.json"
proc_config = load_config("/kaggle/input/datasets/gudagudao/microsam-finetune-1/micro_sam_config.json")

In [ ]:
proc_config['preprocessing']['resize_image'] = False
proc_config['preprocessing']['max_dim'] = 1024
proc_config['preprocessing']['quantization'] = "8bit"
proc_config['preprocessing']['two_channel_merge_mode'] = "passthrough"

with open(data_config) as f:
    data = json.load(f)
    
samples = data["samples"]

images = []
labels = []
for sample in samples:
    label_path = Path(sample["label"]).expanduser()
    
    image_channels = sample["image"]          # dict of channel_name → path
    dic_path = Path(image_channels["dic"]).expanduser()
    fluo_path = image_channels.get("fluorescence")  # None if not present
    fluo_path = Path(fluo_path).expanduser() if fluo_path is not None else fluo_path
    dapi_path = image_channels.get("dapi")        # None if not present
    dapi_path = Path(dapi_path).expanduser() if dapi_path is not None else dapi_path
    
    dic_shift = sample["correct_DIC_shift"]
    proc_config['preprocessing']['correct_DIC_shift'] = dic_shift
    
    labels.append(ImageContainer(label_path, proc_config, is_label=True))
    image = ImageContainer([fluo_path,dapi_path,dic_path], proc_config) if fluo_path is not None and dapi_path is not None else ImageContainer(dic_path, proc_config)
    images.append(image)

## Check loaded data

In [ ]:
# Quick visual sanity check of one preprocessed image / label pair.
import matplotlib.pyplot as plt
from torch_em.util.util import get_random_colors

idx = 5
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
images[idx].display(ax=axes[0], title="image")
labels[idx].display(ax=axes[1], title="labels", cmap=get_random_colors(labels[idx].merge()))
plt.show()

## Data processing

In [ ]:
# Stage 1: preprocess ALL images with ImageContainer and save to disk for the DDP script.
# No train/val split here - candida_multigpu_ais.py does k-fold splitting itself.
# img_{i:03d} zero-padding guarantees sorted() aligns raw <-> label pairs in the script.
import os, numpy as np, tifffile

def to_chw(arr):
    if arr.ndim == 2:
        return np.stack([arr, arr, arr], axis=0)   # 1-channel -> 3-channel CHW
    return arr.transpose(2, 0, 1)                    # HWC -> CHW

out_root  = "/kaggle/working/preprocessed"
raw_dir   = os.path.join(out_root, "raw")
label_dir = os.path.join(out_root, "labels")
os.makedirs(raw_dir,   exist_ok=True)
os.makedirs(label_dir, exist_ok=True)

# `images` and `labels` are the ImageContainer lists built above.
for i, (img, lbl) in enumerate(zip(images, labels)):
    raw = to_chw(img.merge())          # (3, H, W); require_8bit in the loader handles dtype
    lab = lbl.merge()                  # (H, W) instance labels
    tifffile.imwrite(os.path.join(raw_dir,   f"img_{i:03d}.tif"), raw)
    tifffile.imwrite(os.path.join(label_dir, f"img_{i:03d}.tif"), lab)
    print(f"saved img_{i:03d}: raw {raw.shape}, label instances={int(lab.max())}")

print(f"\nDone. raw_dir={raw_dir}  label_dir={label_dir}")

## Device check

In [ ]:
# Confirm both T4s are visible before launching distributed training.
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} {torch.cuda.get_device_name(i)}  cc={torch.cuda.get_device_capability(i)}")

### Live monitoring with TensorBoard

Launch this **before** running the overfit / training cells so the curves update live.
- Overfit: `overfit/loss` plus raw/target/prediction maps (scrub the IMAGES step slider to watch them sharpen).
- Cross-validation: `train/loss`, `validation/loss`, `validation/metric` as one run per fold (`microsam_candida_fold*`).

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir /kaggle/working/overfit_logs

## (Optional) Single-image overfit sanity check

Before the full CV run, confirm the pipeline can memorize one image. Runs on two GPUs. Each GPU logs to its own subfolder. The gradient updates are averaged between the outputs of the two GPUs each step. See documentation for details.

If the loss does **not** collapse toward 0, the bug is in data/model wiring, not generalization.

In [ ]:
# %rm -rf /kaggle/working/overfit_logs/

In [ ]:
# Optional: verify the model can overfit a single image (two GPU, augmentation off).
# After training, rank 0 segments the WHOLE image with the original and the overfitted model
# (both from in-memory weights, no checkpoint reload) and writes a before/after figure next to
# the run's TensorBoard logs: /kaggle/working/overfit_logs/logs/overfit_img_000/overfit_comparison.png.
!python /kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy/candida_overfit_debug.py \
    --raw   /kaggle/working/preprocessed/raw/img_005.tif \
    --label /kaggle/working/preprocessed/labels/img_005.tif \
    --encoder "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm" \
    --decoder "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm_decoder" \
    --model-type vit_b_lm \
    --patch-shape 512 512 \
    --iterations 500 \
    --lr 1e-4 \
    --pass-threshold 0.10 \
    --multi-gpu \
    --num-workers 1 \
    --save-root /kaggle/working/overfit_logs

In [ ]:
# Before/after whole-image comparison from the ALREADY-SAVED checkpoints (no re-training).
# Loads the original model from the encoder/decoder weights and the overfitted model from the
# saved best.pt, runs tiled AIS on the whole image, and writes the figure next to the TB logs.
import sys
sys.path.insert(0, "/kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy")
sys.path.insert(0, "/kaggle/working/micro-sam/")
from candida_overfit_debug import compare_full_image
from IPython.display import Image, display

img_stem  = "img_000"  # the image you overfit (matches the --raw passed to the overfit cell)
ENCODER   = "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm"
DECODER   = "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm_decoder"
SAVE_ROOT = "/kaggle/working/overfit_logs"
name      = f"overfit_{img_stem}"

fig_path = compare_full_image(
    raw_path=f"/kaggle/working/preprocessed/raw/{img_stem}.tif",
    label_path=f"/kaggle/working/preprocessed/labels/{img_stem}.tif",
    model_type="vit_b_lm",
    encoder=ENCODER, decoder=DECODER,
    best_checkpoint=f"{SAVE_ROOT}/checkpoints/{name}/best.pt",
    figure_dir=f"{SAVE_ROOT}/logs/{name}",
    tile_shape=(512, 512), halo=(64, 64),
)
if fig_path:
    display(Image(filename=fig_path))

To download the saved model checkpoints or images. Use save version and download from there. FileLink is broken for use on kaggle.

## Diagnose the overfit NaN: lr, fp16, or DDP?

The multi-GPU overfit went NaN. Three variables were confounded vs the known-good runs: **fp16**, **high lr**, and **DDP/effective-batch**. This cell isolates **precision × learning rate on a single GPU (no DDP)**, reusing the overfit script's own model + loader so the path is faithful. Read the SUMMARY table, then:

| Observation | Conclusion |
|---|---|
| `fp16 lr=1e-2` NaN **and** `fp32 lr=1e-2` NaN | lr `1e-2` is too high for the optimizer itself — divergence, precision-independent. Not an fp16 problem. |
| `fp16 lr=1e-2` NaN **but** `fp32 lr=1e-2` stable | fp16-specific overflow at that lr — fp32 (or bf16) fixes it. |
| `fp16 lr=1e-4`/`1e-5` stable | locates the fp16 stability threshold; pick an lr below it. |

If single-GPU fp16 reproduces the NaN, DDP is exonerated. If it does **not**, the next test adds DDP back.

In [ ]:
# # Controlled test for the overfit NaN: isolate precision x learning rate on ONE GPU (no DDP),
# # reusing the overfit script's own model + single-image loader so the path matches the real run.
# import sys, torch
# sys.path.insert(0, "/kaggle/working/micro-sam")
# sys.path.insert(0, "/kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy")
# import candida_overfit_debug as cod
# from torch_em.loss import DiceBasedDistanceLoss

# RAW     = "/kaggle/working/preprocessed/raw/img_005.tif"
# LABEL   = "/kaggle/working/preprocessed/labels/img_005.tif"
# ENCODER = "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm"
# DECODER = "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm_decoder"
# MODEL_TYPE, PATCH, N_ITERS = "vit_b_lm", (512, 512), 80
# device  = torch.device("cuda:0")
# loss_fn = DiceBasedDistanceLoss(mask_distances_in_bg=True)

# def run_trial(lr, use_amp, n_iters=N_ITERS):
#     torch.manual_seed(0)
#     model = cod._build_model(MODEL_TYPE, ENCODER, DECODER, device); model.train()
#     loader = cod._single_image_loader(RAW, LABEL, PATCH, n_samples=n_iters, is_train=True)
#     opt = torch.optim.AdamW(model.parameters(), lr=lr)
#     scaler = torch.cuda.amp.GradScaler(enabled=use_amp)   # DefaultTrainer uses a GradScaler for fp16
#     first_nan, losses = None, []
#     for it, (x, y) in enumerate(loader):
#         if it >= n_iters: break
#         x, y = x.to(device), y.to(device)
#         opt.zero_grad()
#         with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
#             loss = loss_fn(model(x), y)
#         lval = loss.item(); losses.append(lval)
#         if it % 10 == 0:
#             print(f"    iter {it:3d}  loss={lval}")
#         if not torch.isfinite(loss):
#             first_nan = it; print(f"    iter {it:3d}  loss={lval}  <-- NON-FINITE"); break
#         scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
#     del model, opt; torch.cuda.empty_cache()
#     return first_nan, losses

# trials = [("fp16  lr=1e-2", dict(lr=1e-2, use_amp=True)),
#           ("fp32  lr=1e-2", dict(lr=1e-2, use_amp=False)),
#           ("fp16  lr=1e-4", dict(lr=1e-4, use_amp=True)),
#           ("fp16  lr=1e-5", dict(lr=1e-5, use_amp=True))]

# summary = []
# for name, kw in trials:
#     print(f"\n=== {name} ===")
#     try:
#         nan_it, losses = run_trial(**kw)
#         verdict = f"NaN at iter {nan_it}" if nan_it is not None else f"stable ({N_ITERS} iters, last={losses[-1]:.4f})"
#     except RuntimeError as e:
#         torch.cuda.empty_cache(); verdict = f"ERROR: {str(e)[:80]}"
#     summary.append((name, verdict)); print(f"  -> {verdict}")

# print("\n" + "="*52 + "\nSUMMARY (no DDP, single GPU)\n" + "="*52)
# for name, verdict in summary:
#     print(f"  {name:16s} -> {verdict}")

## Cross-validation

In [ ]:
%rm -rf /kaggle/working/cross_validation/logs/microsam_candida_fold5

In [ ]:
# Cross-validation (leave-one-out, 2x T4 DDP), run ONE FOLD PER PROCESS.
# Running all folds inside a single process leaks DataLoader semaphores / shared memory / NCCL
# process groups across folds on Kaggle, which eventually hangs a rank (NCCL ALLREDUCE timeout,
# typically by the 3rd fold). Launching each fold as its own python process gives a clean teardown
# every time, and a failure in one fold no longer kills the others.
# Per fold: train on 5 images, validate on a deterministic tile grid over the held-out image, then
# run whole-image AIS on it (original vs fine-tuned) -> 2x6 figure + mSA. Models are not kept.
# Outputs accumulate under --save-root (names include the fold index):
#   /kaggle/working/cross_validation/logs/<name>_fold<k>/comparison_<img>.png
import subprocess

SCRIPT = "/kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy/candida_multigpu_ais.py"
N_FOLDS = 6
START_FOLD = 5   # 0-based; resume here. Folds 0 and 1 are done; the 3rd fold is index 2.
common = [
    "--raw-dir", "/kaggle/working/preprocessed/raw",
    "--label-dir", "/kaggle/working/preprocessed/labels",
    "--encoder", "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm",
    "--decoder", "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm_decoder",
    "--save-root", "/kaggle/working/cross_validation",
    "--model-type", "vit_b_lm",
    "--patch-shape", "512", "512",
    "--n-folds", str(N_FOLDS),
    "--batch-size", "1",
    "--lr", "1e-5",
    "--iterations", "1000",
    "--num-workers", "2",
]

for k in range(START_FOLD, N_FOLDS):
    print(f"\n{'='*70}\nLaunching fold {k}/{N_FOLDS - 1} (separate process)\n{'='*70}", flush=True)
    ret = subprocess.run(["python", SCRIPT, *common, "--fold", str(k)])
    if ret.returncode != 0:
        print(f"[fold {k}] exited with code {ret.returncode}; continuing with the next fold.", flush=True)


## Finetune on all data

In [ ]:
# Final model: fine-tune on ALL labeled images across both T4 GPUs (no held-out fold).
# Validation = one fixed cell-rich patch per image (checkpoint selector / LR-plateau signal only;
# the k-fold cell above is what estimates generalization). After training, best.pt is auto-exported
# to a split encoder/decoder pair: <save-root>/<name> + <save-root>/<name>_decoder
# (the pretrained vit_*_lm / vit_*_lm_decoder layout), ready for run_automatic_instance_segmentation.
# Also writes, under --save-root:
#   <name>_final_results.csv               per-image val-patch loss + before/after mSA
#   logs/<name>/comparison_<img>.png       2x6 before/after AIS figure per image
# NOTE: the mSA here is scored on the TRAINING images (a fit/sanity check, NOT generalization).
# The comparison runs by default; pass --no-comparison to skip it (val-patch loss is still written).
!python /kaggle/working/micro-sam/finetuning/specialists/training/light_microscopy/candida_finetune_all.py \
    --raw-dir /kaggle/working/preprocessed/raw \
    --label-dir /kaggle/working/preprocessed/labels \
    --encoder "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm" \
    --decoder "/kaggle/input/models/gudagudao/microsam-vit-b-lm/pytorch/default/1/vit_b_lm_decoder" \
    --save-root /kaggle/working/final_model \
    --name microsam_candida_final \
    --model-type vit_b_lm \
    --patch-shape 512 512 \
    --batch-size 1 \
    --lr 1e-5 \
    --comparison-tile-shape 512 512 \
    --comparison-halo 64 64 \
    --iterations 3000 \
    --early-stopping 20